# Moisturizer Ranking Baseline

## Objective

Build a simple, reproducible baseline that ranks moisturizer
alternatives for adults with acne-prone skin and mild blemishes.

The baseline uses reviewed catalogue information and normalized
ingredient presence/absence data. It returns up to three alternatives
with documented reasons and handles exclusions, missing data
and insufficient candidates explicitly.

Ingredient concentrations are unavailable.

In [15]:
import pandas as pd

## 1. Load and inspect the reviewed product catalogue

Load the catalogue and inspect its dimensions, columns and review
statuses.

Read product codes as strings to preserve leading zeros and allow
matching with the ingredient matrix.

The loaded catalogue contains 152 products and 49 columns.

In [16]:
products = pd.read_csv(
    '~/code/Loub26/Swan/raw_data/moisturizer_catalogue_keep_reviewed_v1.csv',
    dtype={'code': str}
)
products.head()

,code,product_name,brands,france,eu,in_scope,exclusion_reason,is_duplicate,to_check_by_hand,nb_ingredients,...,ingredient_matrix_status,final_review_classification,review_source_file,review_policy_annotation,review_evidence_notes,review_quality_flags,review_quality_followup_needed,moisturizer_inclusion_approved,requires_classification_evidence,catalogue_review_complete
0,4070765004748,Gesichtscreme Niacinamide LSF 30,Balea,False,False,False,NaN,False,False,25,...,Matched,Keep,focused_moisturizer_verification_batch_1_appro...,P02,Exact GTIN confirmed by dm. Facial care cream ...,NaN,False,True,False,True
1,7891010255725,Hydro Boost,Neutrogena,False,False,False,NaN,False,True,19,...,Matched,Keep,manual_moisturizer_review_master_v2.csv,NaN,NaN,NaN,False,True,False,True
2,8436097096367,Crème ultra-hydratante,Byphasse,True,True,True,NaN,False,False,38,...,Matched,Keep,streamlined_moisturizer_screening_batch_1_appr...,P01,Exact EAN identifies Byphasse hyaluronic acid ...,NaN,False,True,False,True
3,5201143734776,ΕΝΥΔΑΤΙΚΗ ΚΡΕΜΑ ΣΕ ΜΟΡΦΗ ΤΖΕΛ,Diadermine,True,True,False,list too short,False,True,1,...,Missing from matrix,Keep,focused_moisturizer_verification_batch_1_appro...,P01,Exact EAN confirms Diadermine Hydralist Aquage...,ingredient_matrix_missing; ingredient_data_fol...,True,True,False,True
4,8684149131485,Ceelini Yoğun Nemlendirici Vücut ve Yüz Kremi ...,Ceelini,False,False,False,list too short,False,False,1,...,Missing from matrix,Keep,focused_moisturizer_verification_batch_1_appro...,P03,Exact barcode confirms Ceelini 50 mL intensive...,ingredient_matrix_missing; ingredient_data_fol...,True,True,False,True


In [17]:
products.shape

(152, 49)

In [18]:
products.columns.to_list()

['code',
 'product_name',
 'brands',
 'france',
 'eu',
 'in_scope',
 'exclusion_reason',
 'is_duplicate',
 'to_check_by_hand',
 'nb_ingredients',
 'coverage',
 'inci_text',
 'has_parfum',
 'has_fragrance_allergen',
 'has_essential_oil',
 'has_acne_active',
 'is_rich',
 'is_light',
 'is_anti_ageing',
 'is_baby',
 'is_face_and_body',
 'category_status',
 'category_status_v1',
 'product_category',
 'review_flags_updated',
 'name_normalized',
 'brand_normalized',
 'final_category',
 'review_decision',
 'review_notes',
 'all_review_flags',
 'deduplication_status',
 'review_status',
 'basic_quality_audit',
 'duplicate_partner_code',
 'duplicate_resolution_action',
 'duplicate_verification_required',
 'duplicate_merged',
 'candidate_status',
 'ingredient_matrix_status',
 'final_review_classification',
 'review_source_file',
 'review_policy_annotation',
 'review_evidence_notes',
 'review_quality_flags',
 'review_quality_followup_needed',
 'moisturizer_inclusion_approved',
 'requires_classifica

## 2. Inspect product eligibility and data readiness

Count the values of the scope, duplicate, approval and data-quality
fields before defining the eligibility filters.

Moisturizer classification approval and ingredient-data readiness
are checked separately.

In [19]:
columns_to_check = [
    'france',
    'in_scope',
    'is_duplicate',
    'moisturizer_inclusion_approved',
    'catalogue_review_complete',
    'review_quality_followup_needed',
    'ingredient_matrix_status'
]

for column in columns_to_check:
    print(column)
    print(products[column].value_counts(dropna=False))
    print()

france
france
False    90
True     62
Name: count, dtype: int64

in_scope
in_scope
False    96
True     56
Name: count, dtype: int64

is_duplicate
is_duplicate
False    149
True       3
Name: count, dtype: int64

moisturizer_inclusion_approved
moisturizer_inclusion_approved
True    152
Name: count, dtype: int64

catalogue_review_complete
catalogue_review_complete
True    152
Name: count, dtype: int64

review_quality_followup_needed
review_quality_followup_needed
False    130
True      22
Name: count, dtype: int64

ingredient_matrix_status
ingredient_matrix_status
Matched                148
Missing from matrix      4
Name: count, dtype: int64



## 3. Select initial ranking candidates

Keep products that are flagged as available in France, within
the project scope, not marked as duplicates, and matched to
the ingredient matrix.

This selection retains 56 products.

In [20]:
eligible_products = products[
    (products['france'] == True)
    & (products['in_scope'] == True)
    & (products['is_duplicate'] == False)
    & (products['ingredient_matrix_status'] == 'Matched')
].copy()

print(eligible_products.shape)

eligible_products[['code', 'product_name', 'brands']].head()

(56, 49)


,code,product_name,brands
2,8436097096367,Crème ultra-hydratante,Byphasse
20,3517360008661,Gelée De Nuit Anti-âge à L'argan Bio,"Léa Nature, So Bio Etic"
21,8809686383559,ISNTREE Hyaluronic Acid Moist Cream,ISNTREE
24,3700646200099,Baume tempête visage (mixte),NividiSkin
28,3662217004867,Crème visage et corps,avril


In [21]:
print('Numer of products selected :', len(eligible_products))

review_columns = [
    'catalogue_review_complete',
    'review_quality_followup_needed'
]

for column in review_columns:
    print(column)
    print(eligible_products[column].value_counts(dropna=False))
    print()

Numer of products selected : 56
catalogue_review_complete
catalogue_review_complete
True    56
Name: count, dtype: int64

review_quality_followup_needed
review_quality_followup_needed
False    50
True      6
Name: count, dtype: int64



## 4. Set aside products requiring quality follow-up

All 56 selected candidates have a completed catalogue review.
However, 6 are flagged with
`review_quality_followup_needed = True`.

Temporarily exclude these products pending additional evidence,
availability checks, or confirmation of their identity or scope.

Retain the six records and their review flags in
`products_to_check` for traceability.

The initial ranking pool contains 50 products.

In [13]:
products_to_check = eligible_products[
    eligible_products['review_quality_followup_needed'] == True
]

products_to_check[
    ['code', 'product_name', 'review_quality_flags']
]

,code,product_name,review_quality_flags
64,20277246,Time control - Crème pour la peau,stronger_evidence_recommended
112,3560070890330,Soin de jour hydratant fondant,stronger_primary_evidence_recommended
114,3337873401243,Crème des reines Riche,historical_discontinuation_flag
128,4020829005037,Crème de jour fluide,historical_discontinuation_flag
148,3600541738508,Miracle Wake Up Crème Anti-Âge défatigant,historical_discontinuation_flag
151,3263855728919,Soin hydratant peaux sèches & sensibles,identity_or_scope_verification_needed; keep_wi...


In [14]:
ranking_products = eligible_products[
    eligible_products['review_quality_followup_needed'] == False
].copy()
ranking_products.shape

(50, 49)

## 5. Load the normalized ingredient matrix

Load the binary ingredient matrix, preserving product codes
as strings.

Each ingredient column records whether that ingredient was
detected in the source data. A value of 1 indicates detection
and 0 indicates no detection.

The loaded matrix contains 253 product records and
252 ingredient columns.

In [22]:
ingredient_matrix = pd.read_csv(
    '~/code/Loub26/Swan/raw_data/ingredient_matrix_inci_normalized_v1.csv',
    dtype={'code': str}
)
print(ingredient_matrix.shape)
ingredient_matrix.head()

(235, 253)


,code,"1,2-hexanediol",acetyl dipeptide-1 cetyl ester,acrylates/c10-30 alkyl acrylate crosspolymer,adenosine,albizia julibrissin bark extract,alcohol,alcohol denat.,algin,allantoin,...,tromethamine,ubiquinone,undecane,urea,vitis vinifera seed oil,xanthan gum,xylitol,xylitylglucoside,zinc gluconate,zinc sulfate
0,00012282,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,00027606,0,0,1,0,0,0,0,0,1,...,0,0,0,1,1,1,0,0,0,0
2,0062600061065,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,06010699,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0
4,0717334243408,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


## 6. Match selected products to their ingredient data

Join the 50 selected products with the ingredient matrix
using the exact product code.

Use a left join to preserve the selected product pool.
Check that product codes are unique in both tables and inspect
the merge indicator to identify any unmatched records.

50 matched products.

In [25]:
ranking_data = ranking_products.merge(
    ingredient_matrix,
    on='code',
    how='left',
    validate='one_to_one',
    indicator=True
)

print(ranking_data.shape)
print(ranking_data['_merge'].value_counts())

(50, 302)
_merge
both          50
left_only      0
right_only     0
Name: count, dtype: int64


### Merge result

All 50 selected products matched the ingredient matrix.
No unmatched records were found.

The merged dataset contains 50 rows and 302 columns.

## 7. Validate ingredient values

Check the ingredient columns for missing values and confirm
that all recorded values are either 0 or 1.

In [27]:
ingredient_columns = ingredient_matrix.columns.drop('code')

ingredient_data = ranking_data[ingredient_columns]

print('Missing values:', ingredient_data.isna().sum().sum())

print(
    'All values are binary:',
    ingredient_data.isin([0, 1]).all().all()
)

Missing values: 0
All values are binary: True


### Ingredient validation result

For the 50 selected products, the ingredient matrix contains
no missing values. All ingredient values are binary (0 or 1).

These checks validate the matrix structure. Source INCI
completeness requires separate verification.

In [29]:
column_to_inspect = [
    'product_name',
    'glycerin',
    'hyaluronic acid',
    'sodium hyaluronate',
    'dimethicone',
    'has_parfum',
    'has_fragrance_allergen',
    'has_essential_oil'
]
ranking_data[column_to_inspect].head(10)

,product_name,glycerin,hyaluronic acid,sodium hyaluronate,dimethicone,has_parfum,has_fragrance_allergen,has_essential_oil
0,Crème ultra-hydratante,1,0,1,0,True,True,True
1,Gelée De Nuit Anti-âge à L'argan Bio,0,0,0,0,True,True,False
2,ISNTREE Hyaluronic Acid Moist Cream,1,1,1,0,False,False,False
3,Baume tempête visage (mixte),1,0,0,0,True,False,False
4,Crème visage et corps,1,0,0,0,True,False,False
5,Hydro Energetic Soin Hydratant Anti-Fatigue 24H,1,0,0,1,True,True,True
6,Crème pour la peau Classic,1,0,0,0,True,True,False
7,Crème de jour anti-rides Q10,1,0,1,1,True,True,False
8,Crème peaux intolérantes,1,0,0,0,False,False,False
9,Ultra Facial Cream,1,0,0,1,False,False,False


## 8. Create a fragrance-related indicator

Combine the existing fragrance, fragrance-allergen and
essential-oil flags into one indicator.

The indicator is True when at least one source flag is True.

In [32]:
ranking_data['fragrance_flag'] = (
    (ranking_data['has_parfum'] == True)
    | (ranking_data['has_essential_oil'] == True)
    | (ranking_data['has_fragrance_allergen'] == True)
)

ranking_data['fragrance_flag'].value_counts(dropna=False)

fragrance_flag
True     44
False     6
Name: count, dtype: int64

## 9. Load the ingredient-function reference

Load the CosIng reference containing INCI names, cosmetic
functions and source information.

In [33]:
ingredient_reference = pd.read_csv(
    '~/code/Loub26/Swan/raw_data/Swan_CosIng_Ingredient_Reference_2026-10-26.csv'
)

print(ingredient_reference.shape)

ingredient_reference[
    ['inci_name', 'functions', 'source_url']
].head()

(1521, 15)


,inci_name,functions,source_url
0,"1,2-HEXANEDIOL",SKIN CONDITIONING | SOLVENT,https://ec.europa.eu/growth/tools-databases/co...
1,1-METHYLHYDANTOIN-2-IMIDE,SKIN CONDITIONING,https://ec.europa.eu/growth/tools-databases/co...
2,2-MERCAPTONICOTINOYL GLYCINE,SKIN CONDITIONING - MISCELLANEOUS,https://ec.europa.eu/growth/tools-databases/co...
3,3-O-ETHYL ASCORBIC ACID,SKIN CONDITIONING,https://ec.europa.eu/growth/tools-databases/co...
4,4-METHYLBENZYLIDENE CAMPHOR,LIGHT STABILIZER | UV ABSORBER | UV FILTER,https://ec.europa.eu/growth/tools-databases/co...


## 10. Prepare ingredient names for matching

Create a matching key by converting INCI names to lowercase
and removing leading and trailing spaces.

Preserve the original INCI names and function information.

In [35]:
ingredient_reference['inci_key'] = (
    ingredient_reference['inci_name']
    .str.lower()
    .str.strip()
)

print (ingredient_reference['functions'].value_counts())

ingredient_reference[
    ['inci_name', 'inci_key', 'functions']
].head()

functions
SKIN CONDITIONING                                                                                     264
SKIN CONDITIONING - EMOLLIENT | SKIN CONDITIONING                                                      55
SURFACTANT - EMULSIFYING | SURFACTANT - CLEANSING                                                      46
SURFACTANT - EMULSIFYING                                                                               36
HAIR CONDITIONING | SKIN CONDITIONING                                                                  30
                                                                                                     ... 
SKIN CONDITIONING - EMOLLIENT | HAIR CONDITIONING | REFATTING | SKIN CONDITIONING                       1
HUMECTANT | FRAGRANCE | SKIN CONDITIONING | PERFUMING                                                   1
FRAGRANCE | SKIN PROTECTING | SMOOTHING                                                                 1
ANTISTATIC | BUFFERING | HUMECTANT |

,inci_name,inci_key,functions
0,"1,2-HEXANEDIOL","1,2-hexanediol",SKIN CONDITIONING | SOLVENT
1,1-METHYLHYDANTOIN-2-IMIDE,1-methylhydantoin-2-imide,SKIN CONDITIONING
2,2-MERCAPTONICOTINOYL GLYCINE,2-mercaptonicotinoyl glycine,SKIN CONDITIONING - MISCELLANEOUS
3,3-O-ETHYL ASCORBIC ACID,3-o-ethyl ascorbic acid,SKIN CONDITIONING
4,4-METHYLBENZYLIDENE CAMPHOR,4-methylbenzylidene camphor,LIGHT STABILIZER | UV ABSORBER | UV FILTER


### Explore the available cosmetic functions

Split combined function labels to count each cosmetic
function separately across the loaded CosIng reference.

In [36]:
individual_functions = (
    ingredient_reference['functions']
    .dropna()
    .str.split('|', regex=False)
    .explode()
    .str.strip()
)

individual_functions.value_counts()

functions
SKIN CONDITIONING                690
SKIN CONDITIONING - EMOLLIENT    239
FRAGRANCE                        181
SURFACTANT - EMULSIFYING         181
HAIR CONDITIONING                178
                                ... 
pH ADJUSTERS                       2
PROPELLANT                         1
SURFACE MODIFIER                   1
OXIDISING                          1
DISPERSING NON-SURFACTANT          1
Name: count, Length: 68, dtype: int64

## 11. Check ingredient-name matching coverage

Compare the ingredient names in the normalized matrix with
the prepared CosIng matching keys.

Count matched and unmatched names. Retain unmatched names
for further inspection.

In [39]:
ingredient_matching = pd.DataFrame({
    'ingredient': ingredient_columns
})

ingredient_matching['inci_key'] = (
    ingredient_matching['ingredient']
    .str.lower()
    .str.strip()
)

ingredient_matching['found_in_reference'] = (
    ingredient_matching['inci_key']
    .isin(ingredient_reference['inci_key'])
)

ingredient_matching['found_in_reference'].value_counts()

found_in_reference
True     251
False      1
Name: count, dtype: int64

### Ingredient-name matching result

251 of the 252 ingredient names match the prepared CosIng
reference keys. One unmatched name requires inspection.

In [40]:
ingredient_matching[ingredient_matching['found_in_reference'] == False ]

,ingredient,inci_key,found_in_reference
157,palmitoyl oligopeptide,palmitoyl oligopeptide,False


In [41]:
ingredient_reference[ingredient_reference['inci_key'].str.contains(
    'palmitoyl',
    na=False
)
                     ][['inci_name', 'functions']]

,inci_name,functions
383,COPPER PALMITOYL HEPTAPEPTIDE-14,SKIN CONDITIONING
462,DIPALMITOYL HYDROXYPROLINE,ANTISTATIC | HAIR CONDITIONING | SKIN CONDITIO...
694,HYDROXYPALMITOYL SPHINGANINE,HAIR CONDITIONING | SKIN CONDITIONING
943,PALMITOYL DIPEPTIDE-7,SKIN CONDITIONING
944,PALMITOYL GRAPEVINE SHOOT EXTRACT,ANTIOXIDANT | SKIN PROTECTING
945,PALMITOYL HEXAPEPTIDE-12,SKIN CONDITIONING
946,PALMITOYL PENTAPEPTIDE-4,SKIN CONDITIONING - MISCELLANEOUS
947,PALMITOYL TETRAPEPTIDE-10,SKIN CONDITIONING
948,PALMITOYL TETRAPEPTIDE-7,SKIN CONDITIONING - MISCELLANEOUS
949,PALMITOYL TRIPEPTIDE-1,SKIN CONDITIONING


### Unmatched ingredient

`palmitoyl oligopeptide` has no exact match in the loaded
reference. Related names were inspected, but no alias
mapping was established. Its functions remain unknown.

### Fragrance indicator result

Among the 50 selected products, 44 have at least one
fragrance-related flag and 6 have none detected.

All 50 products remain in the ranking pool at this stage.

## 12. Check for duplicate INCI keys

Check whether multiple reference records share the same
prepared INCI key before joining ingredient functions.

Duplicate keys could create multiple rows for a single
ingredient during the merge and require inspection.

In [42]:
duplicate_names = ingredient_reference[
    ingredient_reference['inci_key'].duplicated(keep=False)
]

print('Rows with duplicate INCI keys:', len(duplicate_names))

duplicate_names[['inci_name', 'inci_key', 'functions']]

Rows with duplicate INCI keys: 6


,inci_name,inci_key,functions
844,MELALEUCA ALTERNIFOLIA LEAF,melaleuca alternifolia leaf,ABRASIVE | SKIN CONDITIONING
845,MELALEUCA ALTERNIFOLIA LEAF,melaleuca alternifolia leaf,ABRASIVE | SKIN CONDITIONING - MISCELLANEOUS
846,MELALEUCA ALTERNIFOLIA LEAF EXTRACT,melaleuca alternifolia leaf extract,SKIN CONDITIONING | PERFUMING
847,MELALEUCA ALTERNIFOLIA LEAF EXTRACT,melaleuca alternifolia leaf extract,SKIN CONDITIONING - MISCELLANEOUS
1097,POLYMETHYL METHACRYLATE,polymethyl methacrylate,FILM FORMING
1098,POLYMETHYL METHACRYLATE,polymethyl methacrylate,BULKING | FILM FORMING


### Check whether duplicate keys affect the ingredient matrix

Identify duplicate reference keys that also occur in the
normalized ingredient matrix.

In [43]:
relevant_duplicates = duplicate_names[
    duplicate_names['inci_key'].isin(
        ingredient_matching['inci_key']
    )
]

relevant_duplicates[
    ['inci_name', 'inci_key', 'functions']
].sort_values('inci_key')

,inci_name,inci_key,functions


### Duplicate-key check result

Three duplicate INCI keys were found in the reference.
None occurs among the 252 ingredients in the normalized
matrix, so they do not affect the current merge.

## 13. Attach cosmetic functions to the matrix ingredients

Select reference records matching the matrix ingredients,
then attach their functions and source URLs.

Preserve all 252 ingredients, including the unmatched name.

In [46]:
relevant_reference = ingredient_reference[
    ingredient_reference['inci_key'].isin(
        ingredient_matching['inci_key']
    )
]

ingredient_functions = ingredient_matching.merge(
    relevant_reference[['inci_key', 'functions', 'source_url']],
    on='inci_key',
    how='left',
    validate='1:1'
)

print(ingredient_functions.shape)
print('Missing functions:', ingredient_functions['functions'].isna().sum())

(252, 5)
Missing functions: 1


### Function-mapping result

The ingredient-function table contains 252 rows.
251 ingredients have function information from the reference.

Palmitoyl oligopeptide remains unmatched, with unknown functions.

## 14. Identify moisturising ingredients

Select matrix ingredients whose reference functions contain
MOISTURISING.

Seven ingredients match this function.

In [58]:
with pd.option_context('display.max_colwidth', None):
    display(
        ingredient_functions[
            ingredient_functions['functions'].str.contains(
                'moist',
                case=False,
                na=False,
                regex=False
            )
        ][['ingredient', 'functions']]
    )

,ingredient,functions
91,dimethiconol,ANTIFOAMING | SKIN CONDITIONING - EMOLLIENT | MOISTURISING
92,dimethylsilanol hyaluronate,HUMECTANT | MOISTURISING | SKIN CONDITIONING
116,hyaluronic acid,ANTISTATIC | HUMECTANT | MOISTURISING | SKIN CONDITIONING
131,jojoba esters,SKIN CONDITIONING - EMOLLIENT | MOISTURISING | SKIN CONDITIONING | SOOTHING
141,mannitol,BINDING | HUMECTANT | FRAGRANCE | MOISTURISING | SKIN CONDITIONING
142,mel,SKIN CONDITIONING - EMOLLIENT | HUMECTANT | MOISTURISING
237,trehalose,HUMECTANT | MOISTURISING


## 15. Create a MOISTURISING presence indicator

For each selected product, check whether at least one ingredient
labelled MOISTURISING in the reference is detected in its
ingredient list.

In [59]:
moisturising_ingredients = ingredient_functions[
    ingredient_functions['functions'].str.contains(
        'moist',
        case=False,
        na=False,
        regex=False
    )
]['ingredient'].tolist()

ranking_data['moisturising_flag'] = (
    ranking_data[moisturising_ingredients] == 1
).any(axis=1)

ranking_data['moisturising_flag'].value_counts(dropna=False)

moisturising_flag
False    36
True     14
Name: count, dtype: int64

### MOISTURISING indicator result

Among the 50 selected products, 14 contain at least one
ingredient labelled MOISTURISING in the reference.

The other 36 have none of these seven ingredients detected.
This indicator does not measure the overall moisturising
performance of the finished product.

## 16. Define the provisional scoring table

Record the existing placeholder weights from Loredana's
simple_score function in a separate table.

The MOISTURISING indicator remains available separately.
Its weight has not yet been defined.

In [78]:
scoring_table = pd.DataFrame({
    'criterion': ['moisturising', 'acne_active'],
    'weight': [1, 1]
})

scoring_table

,criterion,weight
0,moisturising,1
1,acne_active,1


### Calculate the provisional profile-based score

Apply the scoring-table weights according to the product
signals, skin type and reactivity.

In [81]:
weights = scoring_table.set_index('criterion')['weight'].to_dict()

def calculate_score(data):
    return (
        data['moisturising_flag'].astype(int) * weights['moisturising']
        + data['has_acne_active'].astype(int) * weights['acne_active']
    )

In [82]:
scored_products = ranking_data.copy()

scored_products['score'] = calculate_score(scored_products)

scored_products[
    ['product_name', 'moisturising_flag', 'has_acne_active', 'score']
].head()

,product_name,moisturising_flag,has_acne_active,score
0,Crème ultra-hydratante,False,False,0
1,Gelée De Nuit Anti-âge à L'argan Bio,False,False,0
2,ISNTREE Hyaluronic Acid Moist Cream,True,False,1
3,Baume tempête visage (mixte),False,False,0
4,Crème visage et corps,False,False,0


## 17. Rank products using the baseline score

Rank products by descending baseline score, combining
the MOISTURISING and acne-related ingredient indicators.

Break ties by ascending product code for reproducible results.

In [83]:
ranked_products = scored_products.sort_values(
    by=['score', 'code'],
    ascending=[False, True]
).copy()

ranked_products[
    ['code', 'product_name', 'moisturising_flag', 'has_acne_active', 'score']
].head(3)

,code,product_name,moisturising_flag,has_acne_active,score
13,3401360106994,Sébium sensitive,True,True,2
10,26023717,Anti-âge Soin de jour Q10 Active,False,True,1
12,3178040554931,Expert Rides 3D Activateur Hyaluronique Crème ...,True,False,1


## 18. Add explanations for the baseline score

List detected ingredients labelled MOISTURISING and indicate
whether the catalogue's acne-related ingredient flag is positive.

In [85]:
def explain_score(row):
    reasons = []
    detected_ingredients=[]
    
    for ingredient in moisturising_ingredients:
        if row[ingredient] == 1:
            detected_ingredients.append(ingredient)
    
    if detected_ingredients:
        reasons.append(
            'moisturising ingredients detected: '
            + ', '.join(detected_ingredients)
        )
        
    if row['has_acne_active'] == True:
        reasons.append(
            'Positive acne probe ingredients flag in the catalogue'
        )
        
    if not reasons:
        return 'No positive indicator detected for the baseline score'
    
    return '; '.join(reasons)

ranked_products['reason'] = ranked_products.apply(
    explain_score,
    axis=1
)

with pd.option_context('display.max_colwidth', None):
    display(
        ranked_products[
            ['code', 'product_name', 'reason']
        ].head(3)
    )

,code,product_name,reason
13,3401360106994,Sébium sensitive,moisturising ingredients detected: mannitol; Positive acne probe ingredients flag in the catalogue
10,26023717,Anti-âge Soin de jour Q10 Active,Positive acne probe ingredients flag in the catalogue
12,3178040554931,Expert Rides 3D Activateur Hyaluronique Crème Anti-rides Nuit,moisturising ingredients detected: dimethylsilanol hyaluronate


## 19. Create a callable baseline ranker

Calculate the baseline score for the supplied eligible products.
Rank by descending score and break ties by ascending product code.

Return up to three alternatives with explanations and an explicit
status for empty or insufficient results.

In [86]:
def rank_moisturizers(eligible_data):
    ranked = eligible_data.copy()
    ranked['score'] = calculate_score(ranked)
    
    
    ranked = ranked.sort_values(
        by=['score', 'code'],
        ascending=[False, True]
    )
    
    results = ranked.head(3).copy()
    
    if results.empty:
        return {
            'status': 'no_result',
            'message': 'No eligible products available.',
            'results': [] 
        }
    
    results['reason'] = results.apply(
        explain_score,
        axis=1
    )
    
    if len(results) < 3:
        status = 'insufficient_results'
        message = 'Fewer than three eligible products available.'
    else:
        status = 'ok'
        message = 'Three moisturizer alternatives available.'
        
    return {
        'status': status,
        'message': message,
        'results': results[
            ['code', 'product_name', 'moisturising_flag', 'has_acne_active', 'score', 'reason']
        ].to_dict(orient='records')
    }

## 20. Check result counts and ranking stability

Check responses for zero, one, two and more than three
eligible products.

Verify that shuffling the input rows preserves the ranking.

In [87]:
for number in [0, 1, 2, 5]:
    test_data = ranking_data.head(number)
    result = rank_moisturizers(test_data)
    
    print(
        'Input:', number,
        ' - Status:', result['status'],
        ' - Returned:', len(result['results'])
    )

original_result = rank_moisturizers(ranking_data)

shuffled_data = ranking_data.sample(
    frac=1,
    random_state=42
)

shuffled_result = rank_moisturizers(shuffled_data)

print('Same results:', original_result == shuffled_result)

Input: 0  - Status: no_result  - Returned: 0
Input: 1  - Status: insufficient_results  - Returned: 1
Input: 2  - Status: insufficient_results  - Returned: 2
Input: 5  - Status: ok  - Returned: 3
Same results: True


### Baseline checks passed

The ranker returns:
- No results when the eligible pool is empty.
- All available products when only one or two are eligible.
- Three products when at least three are eligible.

Shuffling the input rows preserves the returned results.

## 21. Export baseline inputs and scoring weights

Save the prepared products, selected MOISTURISING ingredient
names and provisional scoring table for reuse by the ranker.

In [89]:
ranking_data.to_csv(
    "../raw_data/baseline_ranking_data.csv",
    index=False
)

pd.DataFrame({
    "ingredient": moisturising_ingredients
}).to_csv(
    "../raw_data/baseline_moisturising_ingredients.csv",
    index=False
)

scoring_table.to_csv(
    '../raw_data/baseline_scoring_table.csv',
    index=False
)

## Decisions and points for team review

| Section | Decision made | Status |
|---|---|---|
| 3 | Keep products flagged as available in France, in scope, not duplicates and matched to the ingredient matrix. | Applied |
| 4 | Temporarily set aside the 6 products requiring quality follow-up. Retain 50 products for ranking. | Agreed and applied |
| 8 | Combine fragrance, fragrance-allergen and essential-oil flags into one indicator using OR. This combined indicator is currently unused in ranking. | Implemented |
| 11 | Leave palmitoyl oligopeptide unmatched and retain its functions as unknown pending verification. | Applied; verification pending |
| 14–15 | Use ingredients labelled MOISTURISING to build a binary hydration indicator. HUMECTANT without MOISTURISING is outside this selection. Concentrations are unavailable. False means no matching ingredient was detected. | Agreed and implemented; team review pending |
| 16 | Assign +1 to moisturising_flag and +1 to has_acne_active. Store weights in a separate scoring table. | Provisional weights implemented; team review pending |
| 16 | Remove the anti-ageing penalty from the placeholder score. Skin type and reactivity do not affect the current score. | Current implementation; questionnaire alignment pending |
| 17 | Rank by descending score. Break ties by ascending product code for reproducible ordering. | Implemented and tested |
| 18 | Explain positive indicators using detected MOISTURISING ingredient names and the catalogue acne-related flag. | Implemented and tested |
| 19 | Return up to three eligible products. Return one or two available products with insufficient_results status. Return an empty list with no_result status when none are available. | Agreed, implemented and tested |
| 20 | Check empty and insufficient results and ranking stability after shuffling input rows. | Tests passed |
| 21 | Export prepared product data, selected ingredient names and the scoring table for reuse by the Python module. | Completed |
| Notebook 05 | Apply questionnaire scope checks and avoid filters before ranking. Avoid fragrance uses has_parfum only; avoid essential oils uses has_essential_oil. | Pending |